# Downscaling of a Spline
We display a panel of two figures. The top figure contains the realization of a periodic random spline at nominal scale, with a specified period, degree, and delay; the bottom figure contains a version of this spline that is upscaled (*i.e.*, magnified) by the positive integer factor $M\in{\mathbb{N}}+1.$ For each spline, the samples at the integers are indicated by circles and stem lines, while the knots are shown as black dots. The boundaries of one period are highlighted in red.

The top thick black disk represents the value $f_{0}(x)$ taken by the nominal spline at argument $x\in{\mathbb{R}}.$ The bottom thick black disk represents the value $f_{\uparrow M}(x_{\uparrow M})$ taken by the magnified spline at argument $x_{\uparrow M}=M\,x.$ The figure captions suggest that the two values are equal.

In [97]:
# Load the required libraries
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
period = 60 # Period with many divisors
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Minification factors
minifs = [(str(m), m) for m in range(1, period + 1) if 0 == period % m]

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Random periodic cubic spline as a corrupted sine function
f0 = sk.PeriodicSpline1D.from_spline_coeff(np.zeros(period), degree = 3)
def update_spline_coeff (
):
    f0.spline_coeff = np.add(
        rng.standard_normal(period),
        np.array([math.sin(2.0 * np.pi * k / period) for k in range(period)], dtype = float)
    )
update_spline_coeff()

# Plot
def update_plot (
    degree0 = 3,
    delay0 = 0.0,
    minif = len(minifs) // 2,
    degree = 1,
    delay = 0.0,
    flipflop = True
):
    # Update of the spline
    f0.degree = degree0
    f0.delay = delay0

    # Downscaling
    fm = f0.downscaled_projected(minification = minif, degree = degree, delay = delay)

    # Dynamic range
    image = {f0.image(), fm.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))
    
    # Plots
    (fig, ax) = plt.subplots()
    # Spline at the nominal scale
    if 0 < degree0:
        # Continuous function
        x = np.linspace(-1.0, period // minif + 1.0, num = 600 + 1, endpoint = True)
        y = [f0.at(x0 * minif) for x0 in x]
        ax.plot(x, y, lw = 3.0, color = "#e0e0e0")
    else:
        # Function with discontinuities
        x = f0.get_knots()
        ax.plot(
            [(x[0] - 1.0) / minif, x[0] / minif],
            [f0.at(x[0] - 0.5), f0.at(x[0] - 0.5)],
            lw = 3.0,
            color = "#e0e0e0"
        )
        for (k, xk) in enumerate(x):
            if 0 < k:
                ax.plot(
                    [x[k - 1] / minif, xk / minif],
                    [f0.at(xk - 0.5), f0.at(xk - 0.5)],
                    lw = 3.0,
                    color = "#e0e0e0"
                )
        ax.plot(
            [x[-1] / minif, (x[-1] + 1.0) / minif],
            [f0.at(x[-1] + 0.5), f0.at(x[-1] + 0.5)],
            lw = 3.0,
            color = "#e0e0e0"
        )
    # Minified spline
    fm.plot((fig, ax), plotpoints = 600 + 1, plotrange = plotrange)
    # Final display
    plt.show()

# Interactions
flipflop_valid = widgets.Valid(value = True)
randomize_button = widgets.Button(
    description = "[ Click Me to Randomize Data ]",
    layout = widgets.Layout(width = "300px")
)
def update_spline (
    button
):
    update_spline_coeff()
    flipflop_valid.value = not flipflop_valid.value
randomize_button.on_click(update_spline)
degree0_intslider = widgets.IntSlider(
    value = 3,
    min = 0,
    max = max_degree,
    description = "degree0"
)
delay0_floatslider = widgets.FloatSlider(
    value = 0.0,
    min = -max_delay,
    max = max_delay,
    step = 0.05,
    description = "delay0"
)
minif_dropdown = widgets.Dropdown(
    options = minifs,
    value = len(minifs) // 2,
    description = "Minification"
)
degree_intslider = widgets.IntSlider(
    value = 1,
    min = 0,
    max = max_degree,
    description = "degree"
)
delay_floatslider = widgets.FloatSlider(
    value = 0.0,
    min = -max_delay,
    max = max_delay,
    step = 0.05,
    description = "delay"
)
ui = widgets.VBox([
    degree0_intslider,
    delay0_floatslider,
    minif_dropdown,
    degree_intslider,
    delay_floatslider,
    randomize_button
])
out = widgets.interactive_output(
    update_plot,
    {
        "degree0": degree0_intslider,
        "delay0": delay0_floatslider,
        "minif": minif_dropdown,
        "degree": degree_intslider,
        "delay": delay_floatslider,
        "flipflop": flipflop_valid
    }
)
display(ui, out)


Output()